# Antmaze — classical planners (soft_floyd / Dijkstra / A* / greedy)

Pha NHANH: 4 planner cổ điển trên cùng noisy landmark graph, **paired** (mọi planner cùng start/goal per episode), n_test_rollouts=60 → CI hẹp. A* success ≡ Dijkstra (kiểm tra chéo), khác ở latency. Chạy trước; MCTS ở notebook `antmaze_mcts`. Kết quả → plot bằng `scripts/plot_planners.py` (paired bootstrap 95% CI + σ*/collapse).

## 1. Code + env (~10–15 phút lần đầu)

In [ ]:
import os
if os.path.isdir('/kaggle/working/latent_landmarks'):
    !cd /kaggle/working/latent_landmarks && git pull -q origin retrain
else:
    !git clone -q -b retrain https://github.com/Jun1801/latent_landmarks.git /kaggle/working/latent_landmarks
if not os.path.isdir('/kaggle/working/wmag'):
    !git clone -q https://github.com/LunjunZhang/world-model-as-a-graph /kaggle/working/wmag
!bash /kaggle/working/latent_landmarks/repro/setup_kaggle.sh

## 2. Restore checkpoint `antmaze_s221_paper` (set `SLUG`)

In [ ]:
!ls /kaggle/input/
import os, shutil
SLUG = 'PUT-DATASET-SLUG-HERE'          # <-- sửa cho khớp /kaggle/input/
CKPT, ENV = 'antmaze_s221_paper', 'AntMaze-v1'
src = f'/kaggle/input/{SLUG}/experiments/{ENV}/{CKPT}/state'
dst = f'/kaggle/working/experiments/{ENV}/{CKPT}/state'; os.makedirs(dst, exist_ok=True)
for f in ['agent.pt', 'algo.pt']:
    shutil.copy(f'{src}/{f}', f'{dst}/{f}')
print('restored ->', os.listdir(dst))

## 3. Verify GPU + env

In [ ]:
!export PATH=/opt/conda/bin:$PATH; export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-};  conda run -n l3p python -c "import torch,mujoco_py; print('cuda',torch.cuda.is_available())"

## 4. Sanity σ=0 (mọi planner ≈ nhau ≈ soft_floyd clean)

In [ ]:
!export PATH=/opt/conda/bin:$PATH; export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-};  conda run -n l3p python /kaggle/working/latent_landmarks/repro/paper_mcts/eval_ablation.py --env antmaze --resume_ckpt antmaze_s221_paper --sims 100 --latency --planners soft_floyd dijkstra astar greedy --regime e1c --episodes 2 --n_test_rollouts 20 --sigmas 0

## 5. E1a (stochastic) — classical sweep

In [ ]:
!export PATH=/opt/conda/bin:$PATH; export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-};  conda run -n l3p python /kaggle/working/latent_landmarks/repro/paper_mcts/eval_ablation.py --env antmaze --resume_ckpt antmaze_s221_paper --sims 100 --latency --planners soft_floyd dijkstra astar greedy --regime e1a --episodes 3 --n_test_rollouts 60 --out /kaggle/working/exp_out/antmaze_e1a_classical.json --sigmas 0 5 10 20

## 6. E1c (bias) — classical sweep, σ tới 0.5

In [ ]:
!export PATH=/opt/conda/bin:$PATH; export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-};  conda run -n l3p python /kaggle/working/latent_landmarks/repro/paper_mcts/eval_ablation.py --env antmaze --resume_ckpt antmaze_s221_paper --sims 100 --latency --planners soft_floyd dijkstra astar greedy --regime e1c --episodes 3 --n_test_rollouts 60 --out /kaggle/working/exp_out/antmaze_e1c_classical.json --sigmas 0 0.05 0.1 0.15 0.2 0.25 0.3 0.4 0.5

## 7. Lấy JSON về
Tải `antmaze_e1a_classical.json`, `antmaze_e1c_classical.json` → gửi lại (plot chung với MCTS).

In [ ]:
!ls -la /kaggle/working/exp_out/ 2>/dev/null || echo 'chưa có output'